RECON assignment V1

In [9]:
! pip install pydicom

In [ ]:
import numpy as np
import pydicom
from concurrent.futures import ProcessPoolExecutor
import matplotlib.pyplot as plt


# 1) DICOM beolvasás
def load_projections(dicom_path):
    """
    DICOM fájl beolvasása és a projekciók kinyerése.
    A DICOM-ben 3D képtömb van: (frames, rows, cols) = (szögek, sorok, oszlopok).
    Ebből minden frame-ből a detektor középső sorát vesszük ki,
    így kapunk egy 2D sinogramot: (num_angles, num_detectors).
    """
    # DICOM fájl beolvasása
    ds = pydicom.dcmread(dicom_path)

    # Pixeladatok float32 formátumba alakítva
    # Várt alak: (120, 256, 256)
    arr = ds.pixel_array.astype(np.float32)
    print(f"{dicom_path} raw shape:", arr.shape)

    # Ellenőrzés: tényleg 3 dimenziós-e a képtömb
    if arr.ndim != 3:
        raise ValueError(f"Várt shape (frames, rows, cols), de ez jött: {arr.shape}")

    num_frames, rows, cols = arr.shape

    # Vegyük a detektor KÖZÉPSŐ sorát minden frame-ből → (num_frames, cols) = (120, 256)
    mid_row = rows // 2
    projections = arr[:, mid_row, :]

    print("használt projections shape:", projections.shape)  # (120, 256)
    return projections, ds


# 2) Ramp filter 1D minden vetületre (filtered backprojection)
def ramp_filter(projections):
    """
    Egydimenziós ramp filter alkalmazása minden vetületre.
    Ez a filtered backprojection klasszikus előszűrési lépése:
    frekvenciatartományban |ω| szorzót alkalmazunk.
    """
    # projekciók alakja: (szögek száma, detektor elemek száma)
    num_angles, num_detectors = projections.shape

    # Frekvenciatartomány: csak a valós spektrum felét számoljuk (rfftfreq)
    freqs = np.fft.rfftfreq(num_detectors).reshape(1, -1)

    # Ramp filter: abszolút érték → |ω|
    ramp = np.abs(freqs)

    # Vetületek FFT-je detektor irányban
    proj_fft = np.fft.rfft(projections, axis=1)

    # Szűrés frekvenciatartományban
    proj_fft_filtered = proj_fft * ramp

    # Inverz FFT → vissza idősíkba / detektor-síkba
    filtered = np.fft.irfft(proj_fft_filtered, n=num_detectors, axis=1)
    return filtered


# 3) Egy szög hozzájárulása (ezt fogjuk párhuzamosítani)
def backproject_single(args):
    """
    Egyetlen vetület (egy szög) visszavetítése a kép rácsára.
    Ezt a függvényt hívjuk meg párhuzamosan több processzből.
    """
    idx, theta, projection, x_grid, y_grid, det_positions = args

    # Koordináta-transzformáció:
    # x' = x cosθ + y sinθ  → a kép (x, y) pontjait vetítjük a detektor tengelyére.
    t = x_grid * np.cos(theta) + y_grid * np.sin(theta)

    # Interpoláció: a detektor pozíciókról (det_positions) a t értékekre mintavételezünk.
    # t.ravel() → 1D-re lapítjuk a rácsot, majd a végeredményt visszaalakítjuk 2D formára.
    contrib = np.interp(
        t.ravel(),
        det_positions,
        projection,
        left=0.0,
        right=0.0
    ).reshape(x_grid.shape)

    # Ez a mátrix az adott vetület hozzájárulása a rekonstrukcióhoz.
    return contrib


def reconstruct_fbp_parallel(projections, angles_rad, img_size=256):
    """
    Filtered Backprojection (FBP) rekonstrukció párhuzamosítva.
    - projections: (num_angles, num_detectors) sinogram
    - angles_rad: vetületi szögek (radiánban)
    - img_size: kimeneti kép mérete (img_size x img_size)
    """
    num_angles, num_detectors = projections.shape

    # Képrács létrehozása: [-1, 1] × [-1, 1] tartományban
    x = np.linspace(-1.0, 1.0, img_size)
    y = np.linspace(-1.0, 1.0, img_size)
    x_grid, y_grid = np.meshgrid(x, y)

    # Detektor pozíciók: szintén [-1, 1] tartományba skálázva
    det_positions = np.linspace(-1.0, 1.0, num_detectors)

    # Előszűrés ramp filterrel (FBP első lépése)
    filtered_projections = ramp_filter(projections)

    # Argumentumok előkészítése a párhuzamos visszavetítéshez
    # Minden szögre összecsomagoljuk a szükséges adatokat.
    task_args = [
        (i, angles_rad[i], filtered_projections[i], x_grid, y_grid, det_positions)
        for i in range(num_angles)
    ]

    # Rekonstrukciós kép kezdőértéke: nullamatrx
    recon = np.zeros_like(x_grid, dtype=np.float32)

    # Párhuzamos futtatás szögek mentén (processzekkel)
    with ProcessPoolExecutor() as ex:
        # ex.map sorban adja vissza az egyes szögek hozzájárulását (contrib)
        for contrib in ex.map(backproject_single, task_args):
            recon += contrib  # összegezzük az egyes vetületek hozzájárulását

    # Normalizálás: az összegzett képet elosztjuk a vetületek számával
    recon /= num_angles
    return recon


def main():
    """
    Főprogram:
    - két DICOM fájl beolvasása,
    - rekonstrukció mindkettőre,
    - eredmények megjelenítése.
    """
    # --- 1) két DICOM beolvasása ---
    proj1, ds1 = load_projections("jaszczak_lehr-hs_130mm_main_VBProjection_signals.dcm")
    proj2, ds2 = load_projections("jaszczak_lehr-hs_130mm_main_VBProjection_signals_seed31337_120m.dcm")

    # --- 2) szögek kinyerése / feltételezése ---
    # Itt egyszerűsítésként feltételezzük, hogy 0..π között egyenletesen osztott szögeink vannak.
    # num_angles = vetületek száma (frames)
    num_angles = proj1.shape[0]
    angles_rad = np.linspace(0, np.pi, num_angles, endpoint=False)  # párhuzamos nyaláb eset

    # --- 3) rekonstrukció mindkét adatra ---
    recon1 = reconstruct_fbp_parallel(proj1, angles_rad, img_size=256)
    recon2 = reconstruct_fbp_parallel(proj2, angles_rad, img_size=256)

    # --- 4) eredmények megjelenítése ---
    fig, axs = plt.subplots(1, 2)

    # Első rekonstrukció (alap jel)
    axs[0].imshow(recon1, cmap="gray")
    axs[0].set_title("Rekonstrukció - alap jel")
    axs[0].axis("off")

    # Második rekonstrukció (seed31337 – zajosabb verzió)
    axs[1].imshow(recon2, cmap="gray")
    axs[1].set_title("Rekonstrukció - seed31337")
    axs[1].axis("off")

    plt.tight_layout()
    plt.show()




In [ ]:
def load_projections(dicom_path):
    ds = pydicom.dcmread(dicom_path)
    arr = ds.pixel_array.astype(np.float32)
    print(dicom_path, "shape:", arr.shape, "ndim:", arr.ndim)
    return arr, ds

In [ ]:
load_projections("jaszczak_lehr-hs_130mm_main_VBProjection_signals.dcm")

In [ ]:
if __name__ == "__main__":
    main()

![v1](pic_v1.png)